In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/brjapon/cwru-bearing-datasets/CWRU_48k_load_1_CNN_data.npz
/kaggle/input/datasets/brjapon/cwru-bearing-datasets/feature_time_48k_2048_load_1.csv
/kaggle/input/datasets/brjapon/cwru-bearing-datasets/raw/IR021_1_214.mat
/kaggle/input/datasets/brjapon/cwru-bearing-datasets/raw/B014_1_190.mat
/kaggle/input/datasets/brjapon/cwru-bearing-datasets/raw/OR007_6_1_136.mat
/kaggle/input/datasets/brjapon/cwru-bearing-datasets/raw/OR014_6_1_202.mat
/kaggle/input/datasets/brjapon/cwru-bearing-datasets/raw/B007_1_123.mat
/kaggle/input/datasets/brjapon/cwru-bearing-datasets/raw/IR007_1_110.mat
/kaggle/input/datasets/brjapon/cwru-bearing-datasets/raw/B021_1_227.mat
/kaggle/input/datasets/brjapon/cwru-bearing-datasets/raw/Time_Normal_1_098.mat
/kaggle/input/datasets/brjapon/cwru-bearing-datasets/raw/OR021_6_1_239.mat
/kaggle/input/datasets/brjapon/cwru-bearing-datasets/raw/IR014_1_175.mat


In [2]:
import numpy as np
import os
import glob
from scipy.io import loadmat
from scipy.signal import butter, filtfilt

import matplotlib.pyplot as plt
import seaborn as sns
sns.set(style="whitegrid")

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score
)
from sklearn.utils.class_weight import compute_class_weight

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, Model, Input
from tensorflow.keras.callbacks import (
    EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
)
from tensorflow.keras.utils import to_categorical
import pandas as pd
from scipy.stats import kurtosis, skew
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

print(f"TensorFlow version: {tf.__version__}")
print(f"GPU available: {tf.config.list_physical_devices('GPU')}")

2026-04-01 02:09:17.949705: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1775009358.351800      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1775009358.449107      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1775009359.451514      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1775009359.451560      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1775009359.451564      55 computation_placer.cc:177] computation placer alr

TensorFlow version: 2.19.0
GPU available: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU'), PhysicalDevice(name='/physical_device:GPU:1', device_type='GPU')]


## CNN Processings

In [3]:
# ─── Hyperparameters (identical structure to Paderborn notebook) ───
FS            = 48000       # Sampling frequency (Hz) — CWRU drive-end @ 48 kHz
WINDOW_SIZE   = 2048        # Samples per segment
OVERLAP       = WINDOW_SIZE // 2   # 50% overlap
N_CLASSES     = 10          # Number of fault classes
BATCH_SIZE    = 64
EPOCHS        = 50
LEARNING_RATE = 1e-3

# Random seed for reproducibility
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

In [4]:
DATA_DIR = "/kaggle/input/datasets/brjapon/cwru-bearing-datasets/raw/"

# File mapping: filename prefix → (label_id, class_name)
# 10 classes: 1 normal + 3 ball faults + 3 inner race + 3 outer race
FILE_MAP = {
    "Time_Normal": (0, "Normal"),
    "B007":        (1, "Ball-007"),
    "B014":        (2, "Ball-014"),
    "B021":        (3, "Ball-021"),
    "IR007":       (4, "IR-007"),
    "IR014":       (5, "IR-014"),
    "IR021":       (6, "IR-021"),
    "OR007":       (7, "OR-007"),
    "OR014":       (8, "OR-014"),
    "OR021":       (9, "OR-021"),
}

CLASS_NAMES = [v[1] for v in sorted(FILE_MAP.values(), key=lambda x: x[0])]
# print(f"Classes ({N_CLASSES}): {CLASS_NAMES}")

In [5]:
def find_de_key(mat_dict: dict) -> str:
    """
    CWRU .mat files have variable key names for the drive-end signal,
    e.g. 'X097_DE_time', 'X122_DE_time', etc.
    Locate it by checking for 'DE_time' substring.
    """
    for key in mat_dict:
        if "DE_time" in key and not key.startswith("__"):
            return key
    raise KeyError(f"No DE_time key found. Available keys: {list(mat_dict.keys())}")

In [6]:
def extract_signal(mat_content: dict) -> np.ndarray:
    """
    Extract the vibration signal from a loaded .mat file.
    For CWRU: uses the drive-end accelerometer signal (DE_time).
    Returns a 1D float64 numpy array.
    
    NOTE: Same function name as Paderborn notebook for transfer learning compatibility.
    """
    de_key = find_de_key(mat_content)
    signal = mat_content[de_key].squeeze().astype(np.float64)
    return signal

In [7]:
def load_dataset(data_dir: str) -> tuple:
    """
    Load all raw vibration signals from the CWRU dataset.
    
    Returns:
        raw_signals : dict[int, ndarray] — {label_id: 1D signal array}
        label_names : dict[int, str]     — {label_id: class_name}
    
    NOTE: Same function name and return format as Paderborn notebook.
    """
    raw_signals = {}
    label_names = {}
    
    mat_files = [f for f in os.listdir(data_dir) if f.endswith(".mat")]
    if not mat_files:
        raise FileNotFoundError(f"No .mat files found in '{data_dir}'")
    
    # print(f"\n{'='*60}")
    # print("STEP 1 — Loading raw signals")
    # print(f"{'='*60}")
    
    for filename in sorted(mat_files):
        filepath = os.path.join(data_dir, filename)
        
        # Match filename prefix to FILE_MAP
        matched_label = None
        matched_name  = None
        for prefix, (label_id, label_name) in FILE_MAP.items():
            if filename.startswith(prefix):
                matched_label = label_id
                matched_name  = label_name
                break
        
        if matched_label is None:
            print(f"  [SKIP] {filename} — not in FILE_MAP")
            continue
        
        mat = loadmat(filepath)
        signal = extract_signal(mat)
        
        raw_signals[matched_label] = signal
        label_names[matched_label] = matched_name
        
        # print(f"  [OK]   {filename:<35} → class {matched_label:02d} "
        #       f"'{matched_name}'   samples={len(signal):,}")
    
    print(f"\n  Loaded {len(raw_signals)} / {N_CLASSES} classes.\n")
    return raw_signals, label_names

In [8]:
def bandpass_filter(signal: np.ndarray,
                    fs: int,
                    low_hz: float = 100.0,
                    high_hz: float = 5500.0,
                    order: int = 4) -> np.ndarray:
    """
    4th-order Butterworth bandpass filter.
    Removes DC/low-freq drift (< 100 Hz) and high-freq noise (> 5500 Hz).
    
    NOTE: Identical to Paderborn notebook.
    """
    nyq = fs / 2.0
    low  = low_hz  / nyq
    high = high_hz / nyq
    b, a = butter(order, [low, high], btype='band')
    return filtfilt(b, a, signal)


def preprocess_signal(signal: np.ndarray,
                      fs: int = FS,
                      apply_filter: bool = False) -> np.ndarray:
    """
    Full per-signal preprocessing chain:
      1. Remove DC offset   (subtract mean)
      2. Bandpass filter     (100 – 5500 Hz)   [optional]
      3. Z-score normalise   (zero mean, unit variance)
    
    NOTE: Identical to Paderborn notebook.
    """
    # 1. DC removal
    signal = signal - np.mean(signal)
    
    # 2. Bandpass filter
    if apply_filter:
        signal = bandpass_filter(signal, fs)
    
    # 3. Z-score normalisation
    std = signal.std() + 1e-10
    signal = signal / std
    
    return signal


def preprocess_all(raw_signals: dict, fs: int = FS,
                   apply_filter: bool = True) -> dict:
    """
    Apply preprocessing to every class signal.
    
    NOTE: Identical to Paderborn notebook.
    # """
    # print(f"{'='*60}")
    # print("STEP 2 — Preprocessing signals")
    # print(f"{'='*60}")
    
    processed = {}
    for label, signal in raw_signals.items():
        processed[label] = preprocess_signal(signal, fs, apply_filter)
        # print(f"  Class {label:02d}  —  DC removed, filtered, z-scored  "
        #       f"(μ={processed[label].mean():.4f}, σ={processed[label].std():.4f})")
    # print()
    return processed


In [9]:
def segment_signal(signal: np.ndarray,
                   window_size: int = WINDOW_SIZE,
                   overlap: int = None) -> np.ndarray:
    """
    Segment a 1D signal into fixed-length windows with overlap.
    
    Args:
        signal:      1D numpy array
        window_size: number of samples per segment
        overlap:     number of overlapping samples (default: 50%)
    
    Returns:
        segments: array of shape (N, window_size, 1)
    
    NOTE: Identical to Paderborn notebook.
    """
    if overlap is None:
        overlap = window_size // 2
    
    step = window_size - overlap
    n_segments = (len(signal) - window_size) // step + 1
    
    segments = np.zeros((n_segments, window_size, 1), dtype=np.float32)
    for i in range(n_segments):
        start = i * step
        end   = start + window_size
        segments[i, :, 0] = signal[start:end]
    
    return segments

### Common for all models (split->segment(overlapping in only test set)

In [10]:
def split_and_segment(processed_signals: dict,
                      label_names: dict,
                      window_size: int = WINDOW_SIZE,
                      overlap: int = None,
                      train_ratio: float = 0.7,
                      val_ratio: float = 0.15,
                      test_ratio: float = 0.15,
                      augment_train: bool = False,
                      n_augments: int = 2) -> dict:
    """
    1. Split each class's raw signal into train/val/test portions
    2. Segment each portion independently (no leakage)
    3. Optionally augment training segments
    
    Returns dict with keys: X_train, y_train, X_val, y_val, X_test, y_test
    
    NOTE: Same split-before-segment approach as Paderborn notebook.
    """
    print(f"{'='*60}")
    print("STEP 3 — Splitting & Segmenting")
    print(f"{'='*60}")
    
    all_X_train, all_y_train = [], []
    all_X_val,   all_y_val   = [], []
    all_X_test,  all_y_test  = [], []
    
    for label_id in sorted(processed_signals.keys()):
        signal = processed_signals[label_id]
        n = len(signal)
        
        # Split boundaries
        train_end = int(n * train_ratio)
        val_end   = int(n * (train_ratio + val_ratio))
        
        train_signal = signal[:train_end]
        val_signal   = signal[train_end:val_end]
        test_signal  = signal[val_end:]
        
        # Segment each split
        train_segs = segment_signal(train_signal, window_size, overlap)
        val_segs   = segment_signal(val_signal, window_size, overlap=0)
        test_segs  = segment_signal(test_signal, window_size, overlap=0)
        
        # Create labels
        train_labels = np.full(len(train_segs), label_id, dtype=np.int32)
        val_labels   = np.full(len(val_segs),   label_id, dtype=np.int32)
        test_labels  = np.full(len(test_segs),  label_id, dtype=np.int32)
        
        all_X_train.append(train_segs)
        all_y_train.append(train_labels)
        all_X_val.append(val_segs)
        all_y_val.append(val_labels)
        all_X_test.append(test_segs)
        all_y_test.append(test_labels)
        
        name = label_names.get(label_id, f"Class-{label_id}")
        print(f"  Class {label_id:02d} '{name}': "
              f"train={len(train_segs):5d}  val={len(val_segs):4d}  test={len(test_segs):4d}")
    
    # Concatenate all classes
    X_train = np.concatenate(all_X_train, axis=0)
    y_train = np.concatenate(all_y_train, axis=0)
    X_val   = np.concatenate(all_X_val,   axis=0)
    y_val   = np.concatenate(all_y_val,   axis=0)
    X_test  = np.concatenate(all_X_test,  axis=0)
    y_test  = np.concatenate(all_y_test,  axis=0)
    
    # Shuffle training data
    train_idx = np.random.permutation(len(X_train))
    X_train, y_train = X_train[train_idx], y_train[train_idx]
    
    # One-hot encode
    y_train_cat = to_categorical(y_train, N_CLASSES)
    y_val_cat   = to_categorical(y_val,   N_CLASSES)
    y_test_cat  = to_categorical(y_test,  N_CLASSES)
    
    print(f"\n  Summary:")
    print(f"    Train: {X_train.shape}  labels: {y_train_cat.shape}")
    print(f"    Val:   {X_val.shape}  labels: {y_val_cat.shape}")
    print(f"    Test:  {X_test.shape}  labels: {y_test_cat.shape}")
    print()
    
    return {
        'X_train': X_train, 'y_train': y_train, 'y_train_cat': y_train_cat,
        'X_val':   X_val,   'y_val':   y_val,   'y_val_cat':   y_val_cat,
        'X_test':  X_test,  'y_test':  y_test,  'y_test_cat':  y_test_cat,
    }

In [11]:
#Load raw signals for cnn
raw_signals, label_names = load_dataset(DATA_DIR)

# Preprocess all signals
processed_signals = preprocess_all(raw_signals)

# Split → Segment → Augment
data = split_and_segment(
    processed_signals,
    label_names,
    window_size=WINDOW_SIZE,
    augment_train=False,
    n_augments=2
)

X_train, y_train, y_train_cat = data['X_train'], data['y_train'], data['y_train_cat']
X_val,   y_val,   y_val_cat   = data['X_val'],   data['y_val'],   data['y_val_cat']
X_test,  y_test,  y_test_cat  = data['X_test'],  data['y_test'],  data['y_test_cat']


  Loaded 10 / 10 classes.

STEP 3 — Splitting & Segmenting
  Class 00 'Normal': train=  329  val=  35  test=  35
  Class 01 'Ball-007': train=  332  val=  35  test=  35
  Class 02 'Ball-014': train=  331  val=  35  test=  35
  Class 03 'Ball-021': train=  331  val=  35  test=  35
  Class 04 'IR-007': train=  331  val=  35  test=  35
  Class 05 'IR-014': train=  333  val=  35  test=  35
  Class 06 'IR-021': train=  330  val=  35  test=  35
  Class 07 'OR-007': train=  331  val=  35  test=  35
  Class 08 'OR-014': train=  330  val=  35  test=  35
  Class 09 'OR-021': train=  333  val=  35  test=  35

  Summary:
    Train: (3311, 2048, 1)  labels: (3311, 10)
    Val:   (350, 2048, 1)  labels: (350, 10)
    Test:  (350, 2048, 1)  labels: (350, 10)



In [12]:
def build_cnn_model(input_shape: tuple,
                    n_classes: int = N_CLASSES,
                    learning_rate: float = LEARNING_RATE) -> Model:
    """
    1D CNN for bearing fault classification.
    
    Architecture (NO residual blocks — identical to Paderborn notebook):
      - 4 Conv1D blocks: Conv1D → BatchNorm → ReLU → MaxPool1D
      - GlobalAveragePooling1D
      - Dense(128) → Dropout(0.5)
      - Dense(64)  → Dropout(0.3)
      - Dense(n_classes, softmax)
    """
    inp = Input(shape=input_shape, name='input')
    
    # Block 1
    x = layers.Conv1D(32, kernel_size=7, padding='same', name='conv1')(inp)
    x = layers.BatchNormalization(name='bn1')(x)
    x = layers.Activation('relu', name='relu1')(x)
    x = layers.MaxPooling1D(pool_size=2, name='pool1')(x)
    
    # Block 2
    x = layers.Conv1D(64, kernel_size=5, padding='same', name='conv2')(x)
    x = layers.BatchNormalization(name='bn2')(x)
    x = layers.Activation('relu', name='relu2')(x)
    x = layers.MaxPooling1D(pool_size=2, name='pool2')(x)
    
    # Block 3
    x = layers.Conv1D(128, kernel_size=5, padding='same', name='conv3')(x)
    x = layers.BatchNormalization(name='bn3')(x)
    x = layers.Activation('relu', name='relu3')(x)
    x = layers.MaxPooling1D(pool_size=2, name='pool3')(x)
    
    # Block 4
    x = layers.Conv1D(256, kernel_size=3, padding='same', name='conv4')(x)
    x = layers.BatchNormalization(name='bn4')(x)
    x = layers.Activation('relu', name='relu4')(x)
    x = layers.MaxPooling1D(pool_size=2, name='pool4')(x)
    
    # Global pooling + classifier
    x = layers.GlobalAveragePooling1D(name='gap')(x)
    x = layers.Dense(128, activation='relu', name='dense1')(x)
    x = layers.Dropout(0.5, name='drop1')(x)
    x = layers.Dense(64, activation='relu', name='dense2')(x)
    x = layers.Dropout(0.3, name='drop2')(x)
    out = layers.Dense(n_classes, activation='softmax', name='output')(x)
    
    model = Model(inputs=inp, outputs=out, name='CWRU_CNN')
    
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=learning_rate),
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )
    
    return model

In [13]:
# Build the model
input_shape = (WINDOW_SIZE, 1)
model = build_cnn_model(input_shape)
# model.summary()

I0000 00:00:1775009396.194650      55 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13757 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1775009396.200889      55 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13757 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


In [14]:
# Compute class weights to handle imbalance
class_weights_array = compute_class_weight(
    class_weight='balanced',
    classes=np.arange(N_CLASSES),
    y=y_train
)

class_weights = {i: w for i, w in enumerate(class_weights_array)}
# print("Class weights:")
# for i, w in class_weights.items():
#     print(f"  {CLASS_NAMES[i]}: {w:.4f}")


# Callbacks (identical to Paderborn notebook)
callbacks = [
    EarlyStopping(
        monitor='val_loss',
        patience=10,
        restore_best_weights=True,
        verbose=1
    ),
    ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=5,
        min_lr=1e-6,
        verbose=1
    ),
    ModelCheckpoint(
        'best_cwru_cnn.keras',
        monitor='val_accuracy',
        save_best_only=True,
        verbose=1
    )
]

In [15]:
# Train the model
history = model.fit(
    X_train, y_train_cat,
    validation_data=(X_val, y_val_cat),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    class_weight=class_weights,
    callbacks=callbacks,
    verbose=1
)

Epoch 1/50


I0000 00:00:1775009401.323025     126 service.cc:152] XLA service 0x7ab3b00147f0 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1775009401.323056     126 service.cc:160]   StreamExecutor device (0): Tesla T4, Compute Capability 7.5
I0000 00:00:1775009401.323062     126 service.cc:160]   StreamExecutor device (1): Tesla T4, Compute Capability 7.5
I0000 00:00:1775009402.069138     126 cuda_dnn.cc:529] Loaded cuDNN version 91002


 5/52 ━━━━━━━━━━━━━━━━━━━━ 1s 26ms/step - accuracy: 0.1368 - loss: 2.4815

I0000 00:00:1775009407.296091     126 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


52/52 ━━━━━━━━━━━━━━━━━━━━ 0s 99ms/step - accuracy: 0.3606 - loss: 1.8492
Epoch 1: val_accuracy improved from -inf to 0.10857, saving model to best_cwru_cnn.keras
52/52 ━━━━━━━━━━━━━━━━━━━━ 16s 135ms/step - accuracy: 0.3635 - loss: 1.8411 - val_accuracy: 0.1086 - val_loss: 2.4331 - learning_rate: 0.0010
Epoch 2/50
49/52 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - accuracy: 0.7830 - loss: 0.6112
Epoch 2: val_accuracy improved from 0.10857 to 0.12286, saving model to best_cwru_cnn.keras
52/52 ━━━━━━━━━━━━━━━━━━━━ 1s 20ms/step - accuracy: 0.7847 - loss: 0.6042 - val_accuracy: 0.1229 - val_loss: 3.0378 - learning_rate: 0.0010
Epoch 3/50
49/52 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - accuracy: 0.8694 - loss: 0.3275
Epoch 3: val_accuracy improved from 0.12286 to 0.24571, saving model to best_cwru_cnn.keras
52/52 ━━━━━━━━━━━━━━━━━━━━ 1s 19ms/step - accuracy: 0.8694 - loss: 0.3267 - val_accuracy: 0.2457 - val_loss: 3.6926 - learning_rate: 0.0010
Epoch 4/50
49/52 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - accuracy

## SVM Feature extraction

In [16]:
def extract_features(signal):
    features = []
    
    # Time-domain
    rms = np.sqrt(np.mean(signal**2))
    mean = np.mean(signal)
    std = np.std(signal)
    var = np.var(signal)
    peak = np.max(np.abs(signal))
    
    kurt = kurtosis(signal)
    sk = skew(signal)
    
    crest_factor = peak / rms
    impulse_factor = peak / np.mean(np.abs(signal))
    shape_factor = rms / np.mean(np.abs(signal))
    
    # Frequency-domain
    fft_vals = np.abs(np.fft.fft(signal))
    freqs = np.fft.fftfreq(len(signal))
    
    dominant_freq = freqs[np.argmax(fft_vals)]
    spectral_energy = np.sum(fft_vals**2)
    spectral_entropy = -np.sum((fft_vals/np.sum(fft_vals)) * np.log(fft_vals/np.sum(fft_vals) + 1e-10))
    
    features.extend([
        rms, mean, std, var, peak,
        kurt, sk,
        crest_factor, impulse_factor, shape_factor,
        dominant_freq, spectral_energy, spectral_entropy
    ])
    
    return features

In [19]:
def get_svm_features_from_3d(X_3d):
    """
    Converts a 3D CNN input array (N, window_size, 1)
    into a 2D SVM feature array (N, 13).
    """
    feature_matrix = []

    for i in range(len(X_3d)):
        signal_1d = X_3d[i, :, 0]
        features = extract_features(signal_1d)
        feature_matrix.append(features)

    return np.asarray(feature_matrix, dtype=np.float32)

In [20]:
X_train_svm_raw = get_svm_features_from_3d(X_train)
X_val_svm_raw   = get_svm_features_from_3d(X_val)
X_test_svm_raw  = get_svm_features_from_3d(X_test)

print(f"X_train_svm_raw shape: {X_train_svm_raw.shape}") # Expected: (N_train, 13)
print(f"X_val_svm_raw shape:   {X_val_svm_raw.shape}")   # Expected: (N_val, 13)
print(f"X_test_svm_raw shape:  {X_test_svm_raw.shape}")  # Expected: (N_test, 13)

X_train_svm_raw shape: (3311, 13)
X_val_svm_raw shape:   (350, 13)
X_test_svm_raw shape:  (350, 13)


In [21]:
# Initialize the scaler
scaler = StandardScaler()

# FIT on training data, then TRANSFORM training data
X_train_svm = scaler.fit_transform(X_train_svm_raw)

# ONLY TRANSFORM validation and testing data (prevents data leakage)
X_val_svm  = scaler.transform(X_val_svm_raw)
X_test_svm = scaler.transform(X_test_svm_raw)

In [22]:
svm_model = SVC(kernel='rbf', C=10, gamma='scale', probability=True)
svm_model.fit(X_train_svm, y_train)

y_pred_svm = svm_model.predict(X_test_svm)

print("Accuracy:", accuracy_score(y_test, y_pred_svm))

Accuracy: 0.8142857142857143


In [23]:
# Soft voting ensemble: average CNN and SVM probabilities
cnn_test_proba = model.predict(X_test, batch_size=BATCH_SIZE, verbose=0)
svm_test_proba = svm_model.predict_proba(X_test_svm)

ensemble_test_proba = 0.5 * cnn_test_proba + 0.5 * svm_test_proba
ensemble_y_pred = np.argmax(ensemble_test_proba, axis=1)

print("CNN accuracy:", accuracy_score(y_test, np.argmax(cnn_test_proba, axis=1)))
print("SVM accuracy:", accuracy_score(y_test, y_pred_svm))
print("Soft-voting ensemble accuracy:", accuracy_score(y_test, ensemble_y_pred))
print("\nEnsemble classification report:\n")
print(classification_report(y_test, ensemble_y_pred, target_names=CLASS_NAMES))

CNN accuracy: 0.9942857142857143
SVM accuracy: 0.8142857142857143
Soft-voting ensemble accuracy: 0.9942857142857143

Ensemble classification report:

              precision    recall  f1-score   support

      Normal       1.00      1.00      1.00        35
    Ball-007       1.00      0.94      0.97        35
    Ball-014       1.00      1.00      1.00        35
    Ball-021       1.00      1.00      1.00        35
      IR-007       1.00      1.00      1.00        35
      IR-014       1.00      1.00      1.00        35
      IR-021       1.00      1.00      1.00        35
      OR-007       1.00      1.00      1.00        35
      OR-014       0.95      1.00      0.97        35
      OR-021       1.00      1.00      1.00        35

    accuracy                           0.99       350
   macro avg       0.99      0.99      0.99       350
weighted avg       0.99      0.99      0.99       350

